# Fake-News Transformer Review

This notebook reviews the fake-news DistilBERT model and compares it with the classical and deep-learning models when those result files exist.

Training lives in `script/train_fake_news_transformer.py`. Benchmark evaluation lives in `script/run_benchmarks.py`.

## Run First

Run these scripts from PyCharm or from the project root:

```bash
uv run python script/train_fake_news_transformer.py
uv run python script/run_benchmarks.py
```

Benchmark outputs are saved under `artifacts/evaluation/fake_news/fake-news-kaggle/<timestamp>/`.

In [3]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Barderus_Legion\PycharmProjects\TrustNet")
TRANSFORMER_RESULTS_FOLDER = PROJECT_ROOT / "artifacts" / "evaluation" / "fake_news" / "fake-news-kaggle"

transformer_runs = sorted([folder for folder in TRANSFORMER_RESULTS_FOLDER.glob("*") if folder.is_dir()], reverse=True) if TRANSFORMER_RESULTS_FOLDER.exists() else []
latest_transformer_run = transformer_runs[0] if transformer_runs else None
latest_transformer_run

WindowsPath('C:/Users/Barderus_Legion/PycharmProjects/TrustNet/artifacts/evaluation/fake_news/fake-news-kaggle/20260603T003238Z')

## Transformer Benchmark Metrics

This table shows the most recent benchmark metrics for the fake-news transformer model.

In [4]:
if latest_transformer_run is None:
    print("No fake-news transformer benchmark found yet. Run script/run_benchmarks.py after training the model.")
else:
    with (latest_transformer_run / "metrics.json").open("r", encoding="utf-8") as file_handle:
        metrics = json.load(file_handle)
    transformer_table = pd.DataFrame([
        {
            "model_type": "transformer",
            "model": "distilbert_fake_news",
            "accuracy": metrics.get("accuracy"),
            "macro_precision": metrics.get("macro_precision"),
            "macro_recall": metrics.get("macro_recall"),
            "macro_f1": metrics.get("macro_f1"),
            "weighted_f1": metrics.get("weighted_f1"),
            "roc_auc": metrics.get("roc_auc"),
            "log_loss": metrics.get("log_loss"),
        }
    ])
    display(transformer_table)

,model_type,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,roc_auc,log_loss
0,transformer,distilbert_fake_news,0.994543,0.994624,0.994445,0.994531,0.994543,None,0.041712


## All Fake-News Model Comparison

This table combines the latest available classical, deep-learning, and transformer results. If a row is missing, run that model's script first.

In [5]:
comparison_rows = []

baseline_folder = PROJECT_ROOT / "artifacts" / "baselines" / "fake_news"
baseline_runs = sorted([folder for folder in baseline_folder.glob("*") if folder.is_dir()], reverse=True) if baseline_folder.exists() else []
if baseline_runs:
    baseline_metrics = pd.read_csv(baseline_runs[0] / "metrics.csv")
    baseline_metrics["model_type"] = "classical"
    comparison_rows.append(baseline_metrics)

deep_learning_file = PROJECT_ROOT / "artifacts" / "deep_learning" / "fake_news" / "metrics.csv"
if deep_learning_file.exists():
    deep_learning_metrics = pd.read_csv(deep_learning_file)
    deep_learning_metrics["model_type"] = "deep_learning"
    comparison_rows.append(deep_learning_metrics)

if latest_transformer_run is not None:
    with (latest_transformer_run / "metrics.json").open("r", encoding="utf-8") as file_handle:
        metrics = json.load(file_handle)
    comparison_rows.append(pd.DataFrame([{
        "model_type": "transformer",
        "model": "distilbert_fake_news",
        "accuracy": metrics.get("accuracy"),
        "macro_precision": metrics.get("macro_precision"),
        "macro_recall": metrics.get("macro_recall"),
        "macro_f1": metrics.get("macro_f1"),
        "weighted_f1": metrics.get("weighted_f1"),
        "roc_auc": metrics.get("roc_auc"),
    }]))

if not comparison_rows:
    print("No model results found yet.")
else:
    all_results = pd.concat(comparison_rows, ignore_index=True, sort=False)
    columns = ["model_type", "model", "accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "roc_auc"]
    all_results = all_results[[column for column in columns if column in all_results.columns]]
    all_results = all_results.sort_values("macro_f1", ascending=False)
    display(all_results)

,model_type,model,accuracy,macro_precision,macro_recall,macro_f1,weighted_f1,roc_auc
5,transformer,distilbert_fake_news,0.994543,0.994624,0.994445,0.994531,0.994543,None
1,classical,linear_svc_tfidf,0.993481,0.994025,0.992674,0.993328,0.993476,NaN
2,classical,ridge_classifier_tfidf,0.992596,0.993238,0.991660,0.992421,0.992590,NaN
3,deep_learning,text_cnn,0.991388,0.990894,0.991525,0.991205,0.991391,0.999534
0,classical,logistic_regression_tfidf,0.989054,0.989741,0.987920,0.988793,0.989044,NaN
4,deep_learning,bidirectional_lstm,0.986801,0.985502,0.987717,0.986542,0.986816,0.99878


## Discussion Notes

Use this comparison to discuss which model family performed best. If the transformer is best, explain whether the improvement is large enough to justify the extra training cost. If a simpler model is close, that is important because simpler models are easier to train, inspect, and reproduce.